In [ ]:
from pathlib import Path

# The project root, where credentials.json lives. Notebooks run from /notebooks, so we point up
ROOT = Path.home() / "Projects-COD" / "inbox-agent"
CREDENTIALS_FILE = ROOT / "credentials.json"   # The app's ID card (from Google Cloud)
TOKEN_FILE = ROOT / "token.json"               # Your saved login, created on first sign-in

# The permission we ask for: read-only Gmail. Least privilege
SCOPES = ["https://www.googleapis.com/auth/gmail.readonly"]

print("ID card found:", CREDENTIALS_FILE.exists())   # Should be True

In [ ]:
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow

def get_credentials():
    """Return valid Google credentials, signing in through the browser only if needed."""
    creds = None

    # If you've signed in before, reuse the saved login
    if TOKEN_FILE.exists():
        creds = Credentials.from_authorized_user_file(str(TOKEN_FILE), SCOPES)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            # The short-term pass expired: quietly get a new one, no browser needed
            creds.refresh(Request())
        else:
            # First time: open the browser so you can sign in and approve
            flow = InstalledAppFlow.from_client_secrets_file(str(CREDENTIALS_FILE), SCOPES)
            creds = flow.run_local_server(port=0)   # port=0 means pick any free port on your laptop
        # Save the login so next time there's no browser step
        TOKEN_FILE.write_text(creds.to_json())

    return creds

creds = get_credentials()
print("Signed in:", creds.valid)

In [ ]:
from googleapiclient.discovery import build

# Create a connection to Gmail using your login
service = build("gmail", "v1", credentials=creds)

# .execute() actually sends the request. Before that, it's just prepared, like writing a query before running it
profile = service.users().getProfile(userId="me").execute()
inbox = service.users().labels().get(userId="me", id="INBOX").execute()

print("Signed in as:", profile["emailAddress"])
print("Unread emails in inbox:", inbox["messagesUnread"])
print("Unread conversations in inbox:", inbox["threadsUnread"])

In [ ]:
import html

def fetch_unread(service, max_results=10):
    """Return a list of unread inbox emails as simple dictionaries."""

    # Call 1 (LIST): ask Gmail for the IDs of unread inbox emails.
    # q uses the same syntax as Gmail's search bar
    response = service.users().messages().list(
        userId="me", q="is:unread in:inbox", maxResults=max_results
    ).execute()

    # If there are no unread emails, the "messages" key is missing, so default to an empty list
    messages = response.get("messages", [])

    records = []
    for msg in messages:
        # Call 2 (GET): open one email by its ID. Headers only, no body: faster, and we pull only what we need
        full = service.users().messages().get(
            userId="me",
            id=msg["id"],
            format="metadata",
            metadataHeaders=["From", "Subject", "Date"],
        ).execute()

        # Headers arrive as a list of {"name": ..., "value": ...} pairs.
        # Turn them into a lookup table so we can ask for headers["Subject"] directly
        headers = {h["name"]: h["value"] for h in full["payload"].get("headers", [])}

        # Build one clean record. .get(key, default) avoids crashes when something is missing
        records.append({
            "id": full["id"],                                        # Needed later to mark it as read
            "thread_id": full["threadId"],                           # Groups replies into one conversation
            "sender": headers.get("From", "(unknown sender)"),
            "subject": headers.get("Subject", "(no subject)"),
            "date": headers.get("Date", ""),
            "snippet": html.unescape(full.get("snippet", "")),      # Gmail encodes ' as &#39;; this decodes it
            "labels": full.get("labelIds", []),                     # e.g. UNREAD, INBOX, CATEGORY_PROMOTIONS
        })

    return records

In [ ]:
import time

start = time.perf_counter()                      # Time the fetch (latency matters as volume grows)
records = fetch_unread(service, max_results=10)
elapsed = time.perf_counter() - start

print(f"Fetched {len(records)} emails in {elapsed:.1f}s")
for r in records:
    # Trim each field so every email fits on one line
    print(f"{r['sender'][:30]:30} | {r['subject'][:50]}")

# Look at one full record to see all the fields
records[0] if records else "No unread emails"

In [ ]:
print(service.users().messages().list(
        userId="me", q="is:unread in:inbox", maxResults=20
    ).execute())